# T3.3 EIT-001 Environmental Monitoring Tool

READ.ME

This notebook calls the databases and project defined in EIT-003 and handles the import and integration of sensor data into the LCIs.

- Setting the project 
- Relevant sensor data manipulation 
- Integration in foreground database as scenario
- LCA results in DPP format

> The functionalities represent the requirements for EU REED WP3 T3.3 EIT-001 deliverable. Notebook depends on 1) EIT-003 Environmental Performance Model

### 1. Set the project

In [ ]:
import bw2data as bd
import pandas as pd
import json
import bw2analyzer as bwa

from reed.utils import Config

config = Config()
bd.projects.set_current(config.PROJECT_NAME)

bonsai = bd.Database(config.BONSAI_NAME)
reed_db = bd.Database("reed")

### 2. Sensor data integration

#### 2.1 Import sensor data

In [ ]:
from reed.sensordata_analysis import (
    run_sensordata_analysis,
    split_final_df_by_company_event,
    sum_by_reed_tool,
)

monitoring_data_path = config.REED_PATH / "Data" / "MonitoringData" / "Data"
tool_descriptions_path = config.REED_PATH / "Data" / "MonitoringData"

sensordata, *_ = run_sensordata_analysis(
    monitoring_data_path,
    tool_descriptions_path,
)

#### 2.2 Scenario setup for sensor data integation

In [139]:
from reed.swapping_data import (
    find_activities_by_name,
    build_scenarios_from_power_active,
    find_subactivity_by_name_fragment,
    build_consumer_input_scenarios_from_hours_for_activity,
    describe_scenarios,
)

Select which event you want to analyse

In [180]:
company = "goimek"
event_id = "OF10001_G_BQC_S8CF2G"
product = "GOIMEK total"
demand = 1

In [181]:
event_dfs = split_final_df_by_company_event(sensordata)
tool_energy_by_company_and_event = sum_by_reed_tool(
    event_dfs[(company, event_id)]
)

#### 2.3 Inspect analysed sensor input

In [182]:
tool_energy_by_company_and_event

energy_consumption  \
company event_id             reed tool                                         
goimek  OF10001_G_BQC_S8CF2G drilling, soraluce                    20.221889   
                             generic machining, soraluce           48.381444   
                             milling, soraluce                    158.945792   

                                                              hours  \
company event_id             reed tool                                
goimek  OF10001_G_BQC_S8CF2G drilling, soraluce            1.502500   
                             generic machining, soraluce   4.166111   
                             milling, soraluce            12.404722   

                                                          Power_Active  
company event_id             reed tool                                  
goimek  OF10001_G_BQC_S8CF2G drilling, soraluce              13.458828  
                             generic machining, soraluce     11.613095  
                             milling, soraluce               12.813329


Find all processes present in the sensordata in the reed-database

In [183]:
tools = tool_energy_by_company_and_event.index.get_level_values(
    "reed tool"
).unique()
found = {
    tool: find_activities_by_name(reed_db, tool, exact=False)
    for tool in tools
}

Find subactivities called A3 for the product under study

In [184]:
product_activity = reed_db.get(name=product)
a3_activity = find_subactivity_by_name_fragment(
    parent_activity=product_activity,
    name_fragment="A3",
)[0]
soraluce_activity = find_subactivity_by_name_fragment(
    parent_activity=a3_activity,
    name_fragment="soraluce machining",
)[0]
electricity = bonsai.get(
    name="production of electricity (market for)",
    location="GB",
    unit="TJ",
)

Create scenario where alle electricity inputs from the sensordata to the processes identified in found.

In [185]:
scenarios = build_scenarios_from_power_active(
    tool_energy_by_company_and_event,
    found,
    electricity,
)
consumer_scenarios = build_consumer_input_scenarios_from_hours_for_activity(
    tool_energy_by_company_and_event,
    found,
    consumer_activity=soraluce_activity,
)
full_scenario = pd.concat([scenarios, consumer_scenarios])

#### 2.4 Inspect adapted technosphere exchanges

In [186]:
full_scenario_readable = describe_scenarios(full_scenario)
full_scenario_readable

,product_name,product_unit,activity_name,activity_unit,original,goimek_OF10001_G_BQC_S8CF2G
0,production of electricity (market for),TJ,"drilling, soraluce",hour,0.000000,0.000048
1,production of electricity (market for),TJ,"generic machining, soraluce",hour,0.000015,0.000042
2,production of electricity (market for),TJ,"milling, soraluce",hour,0.000000,0.000046
3,"drilling, soraluce",hour,"Soraluce machining, GOIMEK",unit,50.165556,1.502500
4,"generic machining, soraluce",hour,"Soraluce machining, GOIMEK",unit,6.523333,4.166111
5,"milling, soraluce",hour,"Soraluce machining, GOIMEK",unit,5.779167,12.404722


### 3. Footprint Monitoring Output (Finished=0)

During the production stage, indicated by the column "finished=0", the client wants to track total climate impacts and impacts per machining operation. The input data is expected to be aggregated per operation.

In [187]:
from reed.LCA_functions import (
    monitoring_footprint_output,
    monitoring_dpp_output,
)

In [188]:
monitoring_footprint_output_unfinished = monitoring_footprint_output(
    product_activity,
    lcia_method=config.lcia_gwp100,
    company=company,
    event_id=event_id,
    scenario_df=full_scenario,
    scenario_col=f"{company}_{event_id}",
    amount=demand,
)
print(json.dumps(monitoring_footprint_output_unfinished, indent=2))

Exported to monitoring_goimek_OF10001_G_BQC_S8CF2G_2026-08-19.json
{
  "product": "GOIMEK total",
  "total_climate_impact_kg_co2eq": 15801.3698,
  "machining_contributions": [
    {
      "operation": "generic machining, soraluce",
      "score": 53.964326
    },
    {
      "operation": "drilling, soraluce",
      "score": 28.065139
    },
    {
      "operation": "milling, soraluce",
      "score": 211.172082
    },
    {
      "operation": "soraluce machine, Goimek",
      "score": 27.600286
    }
  ]
}


### 4. Final Footprint Monitoring Output: DPP Export (Finished=1)

#### 4.1 DPP output

In [189]:
dpp_output = monitoring_dpp_output(
    product_activity,
    lcia_method=config.lcia_gwp100,
    company=company,
    event_id=event_id,
    scenario_df=full_scenario,
    scenario_col=f"{company}_{event_id}",
    amount=demand,
)
print(json.dumps(dpp_output, indent=2))

Exported to DPP_goimek_OF10001_G_BQC_S8CF2G_2026-08-19.json
{
  "DPP": [
    {
      "company": "goimek",
      "event_id": "OF10001_G_BQC_S8CF2G",
      "product": "GOIMEK total",
      "carbonFootprint": {
        "ProductCarbonFootprints": [
          {
            "PcfCalculationMethods": [
              {
                "PcfCalculationMethod": "ISO 14044"
              }
            ],
            "PCF in kg CO2eq": 15250.4,
            "ProcessName": "GOIMEK total A1",
            "ReferenceValueForCalculation": "unit",
            "QuantityOfMeasureForCalculation": 1,
            "LifeCyclePhases": [
              {
                "LifeCyclePhase": "A1"
              }
            ],
            "ExplanatoryStatement": "Product Carbon Footprint associated with the cradle-to-gate life cycle phases (A1-A3) of the product, calculated using the specific production data provided by the company for this product.",
            "GoodsHandoverAddress": {
              "Country": "ES"
 

### 5. Baseline Referencing

In [ ]:
import bw2analyzer as bwa

In [191]:
goimek_baseline_activity = reed_db.get(
    name="GOIMEK total",
    location="ES",
    unit="unit",
)

In [192]:
bwa.print_recursive_calculation(
    goimek_baseline_activity,
    lcia_method=config.lcia_gwp100,
    amount=1,
    max_level=3,
    cutoff=0.005,
)

Fraction of score | Absolute score | Amount | Activity
0001 | 1.639e+04 |     1 | 'GOIMEK total' (unit, ES, None)
  0.93 | 1.525e+04 |     1 | 'GOIMEK A1: material input' (unit, ES, None)
    1.03 | 1.682e+04 |  5.92 | 'basic iron and steel and of ferro-alloys and first products thereof(market for)' (ton, ES, None)
      -0.0071 | -116.4 | -0.03815 | 'production of basic iron and steel and of ferro-alloys and first products thereof' (ton, AT, 
      -0.0107 |  -175 | -0.06741 | 'production of basic iron and steel and of ferro-alloys and first products thereof' (ton, BE, N
      -0.0296 | -485.3 | -0.1573 | 'production of basic iron and steel and of ferro-alloys and first products thereof' (ton, BR, N
      -0.0932 | -1,527 | -0.5177 | 'production of basic iron and steel and of ferro-alloys and first products thereof' (ton, CN, N
      -0.00615 | -100.8 | -0.02667 | 'production of basic iron and steel and of ferro-alloys and first products thereof' (ton, CZ,
      -0.0833 | -1,365 | -0.